# Advanced FM

Beyond two operators: composite modulators, feedback FM, and the special-purpose generators `asymmetric_fm` and `ssb_am`.

Source: [FM documentation](https://ccrma.stanford.edu/software/snd/snd/fm.html)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import IPython.display as ipd

import pysndlib.clm as clm

SR = 44100

def spectrum(x, n=4096):
    rl = np.copy(x[:n].astype(float)); im = np.zeros(n)
    clm.mus_fft(rl, im, n, 1)
    clm.mus_rectangular2polar(rl, im)
    return np.abs(rl[:n//2])

def show(x, title, fmax=6000):
    f = np.arange(2048) * SR / 4096
    plt.plot(f, spectrum(x), lw=0.8); plt.xlim(0, fmax)
    plt.title(title); plt.show()

## Composite modulators

Two modulators in parallel give a spectrum neither alone would produce (summing their outputs into the carrier's input):

In [ ]:
with clm.Sound('parallel.wav', channels=1, srate=SR):
    car = clm.make_oscil(440.0)
    m1 = clm.make_oscil(440.0)
    m2 = clm.make_oscil(880.0)
    amp = clm.make_env([0, 0, 5, 1, 95, 1, 100, 0], duration=1.0)
    for i in range(SR):
        fm = 2.0 * clm.oscil(m1) + 1.5 * clm.oscil(m2)   # parallel modulators
        clm.outa(i, clm.env(amp) * clm.oscil(car, clm.hz2radians(440.0) * fm))

par, _ = clm.file2ndarray('parallel.wav')
show(par[0], 'parallel modulators (440 + 880 into 440)')

In [ ]:
ipd.Audio('parallel.wav', rate=SR)

A serial (cascade) arrangement modulates the modulator itself -- spectra get much denser:

In [ ]:
with clm.Sound('serial.wav', channels=1, srate=SR):
    car = clm.make_oscil(440.0)
    m1 = clm.make_oscil(440.0)
    m2 = clm.make_oscil(110.0)
    amp = clm.make_env([0, 0, 5, 1, 95, 1, 100, 0], duration=1.0)
    for i in range(SR):
        fm = clm.oscil(m1, clm.oscil(m2) * clm.hz2radians(110.0) * 2.0)  # m2 modulates m1
        clm.outa(i, clm.env(amp) * clm.oscil(car, fm * clm.hz2radians(440.0) * 2.0))

ser, _ = clm.file2ndarray('serial.wav')
show(ser[0], 'serial (cascade) modulators')

In [ ]:
ipd.Audio('serial.wav', rate=SR)

## Feedback FM

Feeding the carrier's own delayed output back into its modulator produces the "laser/zap" timbres (the spectrum gains every harmonic as the feedback approaches instability):

In [ ]:
with clm.Sound('feedback.wav', channels=1, srate=SR):
    car = clm.make_oscil(220.0)
    fb = 0.0
    amp = clm.make_env([0, 0, 5, 1, 95, 1, 100, 0], duration=1.0)
    for i in range(SR):
        v = clm.oscil(car, fb)
        fb = 0.9 * v          # feed the previous output back in
        clm.outa(i, clm.env(amp) * v)

fbk, _ = clm.file2ndarray('feedback.wav')
show(fbk[0], 'feedback FM: rich, dense harmonics')

In [ ]:
ipd.Audio('feedback.wav', rate=SR)

## asymmetric_fm

`asymmetric_fm` shapes the spectrum asymmetrically around the carrier (the classic "single-sideband-ish" FM sound):

In [ ]:
with clm.Sound('afm.wav', channels=1, srate=SR):
    a = clm.make_asymmetric_fm(440.0, r=1.0, ratio=1.0)
    amp = clm.make_env([0, 0, 5, 1, 95, 1, 100, 0], duration=1.0)
    idx = clm.make_env([0, 0, 100, 2], duration=1.0)     # the index is a per-sample input
    for i in range(SR):
        clm.outa(i, clm.env(amp) * clm.asymmetric_fm(a, clm.env(idx)))

afm, _ = clm.file2ndarray('afm.wav')
show(afm[0], 'asymmetric_fm')

In [ ]:
ipd.Audio('afm.wav', rate=SR)

## ssb_am: single-sideband AM

`ssb_am` is sine-summation-style amplitude modulation -- odd harmonics in a single sideband, good for clarinet-like timbres:

In [ ]:
with clm.Sound('ssb.wav', channels=1, srate=SR):
    s = clm.make_ssb_am(440.0, order=40)
    amp = clm.make_env([0, 0, 5, 1, 95, 1, 100, 0], duration=1.0)
    inp = clm.make_env([0, 0, 100, 1], duration=1.0)   # the input sweeps the spectrum up
    for i in range(SR):
        clm.outa(i, clm.env(amp) * clm.ssb_am(s, clm.env(inp)))

ssbd, _ = clm.file2ndarray('ssb.wav')
show(ssbd[0], 'ssb_am')

In [ ]:
ipd.Audio('ssb.wav', rate=SR)

## Summary

- Parallel/serial modulator arrangements multiply spectral possibilities
- Feedback FM (previous output into the input) gives dense harmonic zaps
- `asymmetric_fm` and `ssb_am` are ready-made spectral shapers

Next: [the waveform generator zoo](09_Waveforms.ipynb)